# Forex Prediction Model Notebook

In [18]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns


In [19]:

# Load the dataset
file_path = 'EURUSD30.csv'  # Adjust this path as needed
data = pd.read_csv(file_path, header=None)

# Rename columns appropriately
data.columns = ['Date', 'Time', 'Open', 'High', 'Low', 'Close', 'Volume']

# Combine 'Date' and 'Time' into a single datetime column
data['DateTime'] = pd.to_datetime(data['Date'] + ' ' + data['Time'], format='%Y.%m.%d %H:%M')

# Set 'DateTime' as the index and drop the original 'Date' and 'Time' columns
data.set_index('DateTime', inplace=True)
data = data[['Open', 'High', 'Low', 'Close', 'Volume']]

# Feature Engineering: Add Moving Averages and Lag Features
data['SMA_10'] = data['Close'].rolling(window=10).mean()
data['SMA_50'] = data['Close'].rolling(window=50).mean()
data['Lag_1'] = data['Close'].shift(1)


In [20]:


# Drop NaN values created by rolling windows
data.dropna(inplace=True)

# Prepare features and target variable
X = data[['Open', 'High', 'Low', 'Volume', 'SMA_10', 'SMA_50', 'Lag_1']]
y = data['Close']

# Split the data into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [21]:

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [22]:
# Train the model and perform cross-validation
model = RandomForestRegressor(n_estimators=100, random_state=42)
cross_val_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')


In [23]:
# Calculate mean and standard deviation of cross-validation scores
mean_cv_score = cross_val_scores.mean()
std_cv_score = cross_val_scores.std()

In [24]:


print('Mean R-squared score (cross-validation):', mean_cv_score)
print('Standard deviation of R-squared scores (cross-validation):', std_cv_score)
    

Mean R-squared score (cross-validation): 0.9999549726009114
Standard deviation of R-squared scores (cross-validation): 1.1802895553654548e-06


In [25]:
def predict_future_prices_for_days(model, data, num_predictions=5, interval='D'):
    future_predictions = []
    future_dates = pd.date_range(start=data.index[-1], periods=num_predictions + 1, freq=interval)[1:]
    last_row = data.iloc[-1].copy()  # Get the last row as the starting point

    for date in future_dates:
        # Create a feature vector from the last known data
        features = np.array([
            last_row['Open'],
            last_row['High'],
            last_row['Low'],
            last_row['Volume'],
            last_row['SMA_10'],
            last_row['SMA_50'],
            last_row['Lag_1']
        ]).reshape(1, -1)

        # Scale the features
        features_scaled = scaler.transform(features)

        # Predict the next close price
        predicted_price = model.predict(features_scaled)[0]
        future_predictions.append((date, predicted_price))

        # Update the last_row with the predicted value for next iteration
        last_row['Lag_1'] = predicted_price
        last_row['Close'] = predicted_price
        last_row['SMA_10'] = (data['Close'].iloc[-9:].sum() + predicted_price) / 10
        last_row['SMA_50'] = (data['Close'].iloc[-49:].sum() + predicted_price) / 50

    return future_predictions

In [26]:
model.fit(X_train_scaled, y_train)
future_predictions = predict_future_prices_for_days(model, data, num_predictions=5, interval='D')


c:\Users\agbec\anaconda3\envs\mypy3v\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\agbec\anaconda3\envs\mypy3v\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\agbec\anaconda3\envs\mypy3v\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\agbec\anaconda3\envs\mypy3v\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\agbec\anaconda3\envs\mypy3v\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [27]:
future_prices_with_dates = predict_future_prices_with_dates(model, data, num_predictions=50)
for date, price in future_prices_with_dates:
    print(f"Predicted close price on {date}: {price:.5f}")


C:\Users\agbec\AppData\Local\Temp\ipykernel_54404\1511359596.py:7: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  future_dates = pd.date_range(start=data.index[-1], periods=num_predictions + 1, freq=interval)[1:]
c:\Users\agbec\anaconda3\envs\mypy3v\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\agbec\anaconda3\envs\mypy3v\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\agbec\anaconda3\envs\mypy3v\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\agbec\anaconda3\envs\mypy3v\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted 

Predicted close price on 2024-10-31 22:30:00: 1.08722
Predicted close price on 2024-10-31 23:00:00: 1.08721
Predicted close price on 2024-10-31 23:30:00: 1.08721
Predicted close price on 2024-11-01 00:00:00: 1.08721
Predicted close price on 2024-11-01 00:30:00: 1.08721
Predicted close price on 2024-11-01 01:00:00: 1.08721
Predicted close price on 2024-11-01 01:30:00: 1.08721
Predicted close price on 2024-11-01 02:00:00: 1.08721
Predicted close price on 2024-11-01 02:30:00: 1.08721
Predicted close price on 2024-11-01 03:00:00: 1.08721
Predicted close price on 2024-11-01 03:30:00: 1.08721
Predicted close price on 2024-11-01 04:00:00: 1.08721
Predicted close price on 2024-11-01 04:30:00: 1.08721
Predicted close price on 2024-11-01 05:00:00: 1.08721
Predicted close price on 2024-11-01 05:30:00: 1.08721
Predicted close price on 2024-11-01 06:00:00: 1.08721
Predicted close price on 2024-11-01 06:30:00: 1.08721
Predicted close price on 2024-11-01 07:00:00: 1.08721
Predicted close price on 202

c:\Users\agbec\anaconda3\envs\mypy3v\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\agbec\anaconda3\envs\mypy3v\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\agbec\anaconda3\envs\mypy3v\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\agbec\anaconda3\envs\mypy3v\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\agbec\anaconda3\envs\mypy3v\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\agbec\anaconda3\envs\mypy3v\lib